In [ ]:
# ====================================================
# Continue code - prepare the model and start training
# ====================================================

import os
import cv2
import torch
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import albumentations as A
from tqdm import tqdm

# =========================
# FIXED SEED
# =========================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# =========================
# FREE GPU MEMORY
# =========================
torch.cuda.empty_cache()

# =========================
# CONFIG - STAGE 4 (20 Epochs)
# =========================

class_names = ["helmet", "head", "body", "vest"]
NUM_CLASSES = len(class_names) + 1

# Final settings
BATCH_SIZE = 4  # For Colab GPU
EPOCHS = 20

# Very low Learning Rates for Fine-tuning
BACKBONE_LAYER3_LR = 0.00002
BACKBONE_LAYER4_LR = 0.00002
RPN_HEAD_LR = 0.00005
ROI_HEAD_LR = 0.00005

WEIGHT_DECAY = 0.0005
MOMENTUM = 0.9

# Scheduler
LR_FACTOR = 0.5
LR_PATIENCE = 3
MIN_LR = 1e-9

GRAD_CLIP_MAX_NORM = 0.3
EARLY_STOPPING_PATIENCE = 6  # For 20 epochs

# Save paths
SAVE_DIR = "/content/drive/MyDrive/ppe_models"
os.makedirs(SAVE_DIR, exist_ok=True)

# Load the previous model (if available)
LOAD_PRETRAINED = True
PRETRAINED_MODEL_PATH = os.path.join(SAVE_DIR, "faster_rcnn_best_stage3.pth")

# =========================
# AUGMENTATION - Stage 4
# =========================
transform = A.Compose(
    [
        A.Resize(height=640, width=640),
        A.HorizontalFlip(p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.2),
        A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.05, rotate_limit=5, 
                          border_mode=cv2.BORDER_CONSTANT, p=0.2),
    ],
    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"],
        min_visibility=0.3,
        check_each_transform=True,
        min_area=25,
    )
)

val_transform = A.Compose(
    [
        A.Resize(height=640, width=640),
    ],
    bbox_params=A.BboxParams(format="pascal_voc", label_fields=["labels"])
)

# =========================
# DATASET CLASS
# =========================
class CustomDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_list, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_list = image_list
        self.transform = transform

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        img_path = os.path.join(self.image_dir, img_name)
        
        # Find the corresponding label file
        label_name = img_name.replace(".jpg", ".txt").replace(".png", ".txt").replace(".jpeg", ".txt")
        label_path = os.path.join(self.label_dir, label_name)

        image = cv2.imread(img_path)
        if image is None:
            image = np.zeros((640, 640, 3), dtype=np.uint8)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls = int(parts[0])
                    xmin = float(parts[1])
                    ymin = float(parts[2])
                    xmax = float(parts[3])
                    ymax = float(parts[4])

                    xmin = max(0, min(width, xmin))
                    ymin = max(0, min(height, ymin))
                    xmax = max(0, min(width, xmax))
                    ymax = max(0, min(height, ymax))

                    if xmax <= xmin or ymax <= ymin:
                        continue

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(cls + 1)

        if self.transform:
            if len(boxes) > 0:
                augmented = self.transform(image=image, bboxes=boxes, labels=labels)
                image = augmented["image"]
                boxes = augmented["bboxes"]
                labels = augmented["labels"]

                valid_boxes = []
                valid_labels = []
                for box, label in zip(boxes, labels):
                    if len(box) == 4:
                        xmin, ymin, xmax, ymax = box
                        if (xmax > xmin + 1 and ymax > ymin + 1 and 
                            xmin >= 0 and ymin >= 0 and 
                            xmax <= image.shape[1] and ymax <= image.shape[0]):
                            valid_boxes.append(box)
                            valid_labels.append(label)
                boxes = valid_boxes
                labels = valid_labels
            else:
                augmented = self.transform(image=image, bboxes=[], labels=[])
                image = augmented["image"]

        image = torch.from_numpy(image).float().permute(2, 0, 1) / 255.0

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std

        if len(boxes) == 0:
            target = {
                "boxes": torch.zeros((0, 4), dtype=torch.float32),
                "labels": torch.zeros((0,), dtype=torch.int64)
            }
        else:
            target = {
                "boxes": torch.tensor(boxes, dtype=torch.float32),
                "labels": torch.tensor(labels, dtype=torch.int64)
            }

        return image, target

# =========================
# DATA LOADING
# =========================
def filter_images_with_boxes(image_dir, label_dir, image_list):
    valid_images = []
    total_boxes = 0
    
    for img in image_list:
        label_name = img.replace(".jpg", ".txt").replace(".png", ".txt").replace(".jpeg", ".txt")
        label_path = os.path.join(label_dir, label_name)
        
        if os.path.exists(label_path):
            with open(label_path) as f:
                lines = f.readlines()
                if len(lines) > 0:
                    valid_images.append(img)
                    total_boxes += len(lines)
    
    print(f"  Found {len(valid_images)} valid images with {total_boxes} total boxes")
    return valid_images

def collate_fn(batch):
    return tuple(zip(*batch))

# =========================
# CREATING DATALOADERS
# =========================
train_images = [f for f in os.listdir(IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]
val_images = [f for f in os.listdir(VAL_IMAGE_DIR) if f.endswith((".jpg", ".png", ".jpeg"))]

print("\n" + "="*70)
print("📊 FINAL DATA LOADING")
print("="*70)

print(f"\n  Total training images: {len(train_images)}")
print(f"  Total validation images: {len(val_images)}")

print("\nLoading training data...")
train_images = filter_images_with_boxes(IMAGE_DIR, LABEL_DIR, train_images)

print("\nLoading validation data...")
val_images = filter_images_with_boxes(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images)

print(f"\n✅ Final counts:")
print(f"  Training samples: {len(train_images)}")
print(f"  Validation samples: {len(val_images)}")

if len(train_images) == 0:
    raise ValueError("❌ No training images found!")
if len(val_images) == 0:
    print("⚠️ No validation images found! Using 20% of training data for validation...")
    val_images = train_images[:int(len(train_images)*0.2)]

train_dataset = CustomDataset(IMAGE_DIR, LABEL_DIR, train_images, transform)
val_dataset = CustomDataset(VAL_IMAGE_DIR, VAL_LABEL_DIR, val_images, val_transform)

print("\nCreating DataLoaders...")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True
)

print("✅ DataLoaders created successfully!")

# =========================
# MODEL SETUP
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Using device: {device}")
if device.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

torch.cuda.empty_cache()

model = fasterrcnn_resnet50_fpn(weights=None)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

# =========================
# LOAD PRETRAINED WEIGHTS (if exists)
# =========================
if LOAD_PRETRAINED and os.path.exists(PRETRAINED_MODEL_PATH):
    print("\n" + "="*70)
    print("LOADING BEST MODEL FROM STAGE 3")
    print("="*70)
    try:
        state_dict = torch.load(PRETRAINED_MODEL_PATH, map_location=device)
        model.load_state_dict(state_dict, strict=False)
        print(f"  ✅ Successfully loaded model from: {PRETRAINED_MODEL_PATH}")
        print(f"  📊 This model achieved mAP: 0.2961 in Stage 3")
        print(f"  🚀 Continuing training with very low LR for final refinement")
    except Exception as e:
        print(f"  ❌ Failed to load model: {e}")
        print("  🔄 Starting from scratch instead")
        LOAD_PRETRAINED = False
else:
    print(f"\n⚠️ Model not found at: {PRETRAINED_MODEL_PATH}")
    print("  🔄 Starting from scratch")

model.to(device)

# =========================
# LAYER FREEZING - ONLY layer3 and layer4
# =========================
print("\n" + "="*70)
print("LAYER FREEZING CONFIGURATION - STAGE 4")
print("="*70)

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

print("\n📌 Backbone - ONLY layer3 and layer4 fine-tuned:")
layer3_count = 0
layer4_count = 0

for name, param in model.backbone.body.named_parameters():
    if 'layer3' in name:
        param.requires_grad = True
        layer3_count += 1
    elif 'layer4' in name:
        param.requires_grad = True
        layer4_count += 1
    else:
        param.requires_grad = False

print(f"  ✅ layer3 unfrozen: {layer3_count} parameters")
print(f"  ✅ layer4 unfrozen: {layer4_count} parameters")
print(f"  🔒 layer1, layer2, stem: FROZEN")

print("\n📌 RPN - ALL layers fine-tuned:")
rpn_count = 0
for param in model.rpn.parameters():
    param.requires_grad = True
    rpn_count += 1
print(f"  Unfrozen: {rpn_count} parameters")

print("\n📌 ROI Heads - ALL layers fine-tuned:")
roi_count = 0
for param in model.roi_heads.parameters():
    param.requires_grad = True
    roi_count += 1
print(f"  Unfrozen: {roi_count} parameters")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "-"*70)
print(f"📊 Total parameters: {total_params:,}")
print(f"📊 Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print("="*70 + "\n")

# =========================
# OPTIMIZER - AdamW
# =========================
optimizer_params = [
    {
        'params': [p for n, p in model.backbone.body.named_parameters() 
                  if 'layer3' in n and p.requires_grad],
        'lr': BACKBONE_LAYER3_LR,
    },
    {
        'params': [p for n, p in model.backbone.body.named_parameters() 
                  if 'layer4' in n and p.requires_grad],
        'lr': BACKBONE_LAYER4_LR,
    },
    {
        'params': model.rpn.parameters(),
        'lr': RPN_HEAD_LR,
    },
    {
        'params': model.roi_heads.parameters(),
        'lr': ROI_HEAD_LR,
    },
]

optimizer = torch.optim.AdamW(
    optimizer_params,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999)
)

# Cosine Annealing Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS,
    eta_min=1e-9
)

metric = MeanAveragePrecision()

# =========================
# EARLY STOPPING
# =========================
class EarlyStopping:
    def __init__(self, patience=6, mode='max', min_delta=0.0005):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = -float('inf') if mode == 'max' else float('inf')
        self.mode = mode
        self.early_stop = False

    def step(self, score):
        if self.mode == 'max':
            is_better = score > self.best_score + self.min_delta
        else:
            is_better = score < self.best_score - self.min_delta

        if is_better:
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        return False

    def get_status(self):
        return {
            'counter': self.counter,
            'patience': self.patience,
            'best_score': self.best_score,
            'remaining': self.patience - self.counter
        }

early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, mode='max', min_delta=0.0005)

# =========================
# TRAINING LOOP - STAGE 4 (20 Epochs)
# =========================
print("\n" + "="*70)
print("🚀 STARTING STAGE 4 TRAINING (20 EPOCHS - FINAL REFINEMENT)")
print("="*70)
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Total Epochs: {EPOCHS}")
print(f"\n✅ Configuration:")
print(f"  - ONLY layer3+4 unfrozen (layer1+2 frozen)")
print(f"  - AdamW optimizer")
print(f"  - Cosine Annealing scheduler")
print(f"  - Early Stopping Patience: 6")
print(f"\n📊 Learning Rates:")
print(f"  - Backbone layer3: {BACKBONE_LAYER3_LR}")
print(f"  - Backbone layer4: {BACKBONE_LAYER4_LR}")
print(f"  - RPN: {RPN_HEAD_LR}")
print(f"  - ROI Heads: {ROI_HEAD_LR}")
print(f"\n✅ Models will be saved to: {SAVE_DIR}")
print("="*70 + "\n")

best_map = 0
accumulation_steps = 2

for epoch in range(EPOCHS):
    print(f"\n{'='*70}")
    print(f"🚀 STAGE 4 - EPOCH {epoch+1}/{EPOCHS}")
    print(f"{'='*70}")
    
    # -----------------
    # TRAIN PHASE
    # -----------------
    model.train()
    train_loss = 0
    train_losses = {}
    train_num_batches = 0
    
    torch.cuda.empty_cache()

    pbar = tqdm(train_loader, desc=f"Training", unit='batch', 
                ncols=100, position=0, leave=True)
    
    optimizer.zero_grad()
    
    for batch_idx, (images, targets) in enumerate(pbar):
        try:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
            
            if torch.isnan(loss):
                print(f"\n⚠️ NaN detected at batch {batch_idx+1}! Skipping...")
                continue
            
            loss = loss / accumulation_steps
            loss.backward()

            train_loss += loss.item() * accumulation_steps
            for k, v in loss_dict.items():
                if k not in train_losses:
                    train_losses[k] = 0
                train_losses[k] += v.item()
            train_num_batches += 1

            if (batch_idx + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
                optimizer.step()
                optimizer.zero_grad()

            pbar.set_postfix({
                'loss': f'{loss.item() * accumulation_steps:.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.8f}'
            })
                
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(f"\n⚠️ CUDA OOM at batch {batch_idx}. Clearing cache...")
                torch.cuda.empty_cache()
                continue
            else:
                print(f"\n❌ Error in batch {batch_idx}: {e}")
                continue

    if train_num_batches == 0:
        print("\n❌ No batches were processed! Check your data.")
        continue

    if (batch_idx + 1) % accumulation_steps != 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_MAX_NORM)
        optimizer.step()
        optimizer.zero_grad()

    pbar.close()

    train_loss_avg = train_loss / train_num_batches if train_num_batches > 0 else 0
    train_losses_avg = {k: v / train_num_batches for k, v in train_losses.items()} if train_num_batches > 0 else {}

    # -----------------
    # VALIDATION PHASE
    # -----------------
    print(f"\n🔄 Validating...")
    model.eval()
    metric.reset()

    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc="Validation", unit='batch', ncols=100, position=0, leave=True)
        for images, targets in val_pbar:
            try:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                preds = model(images)
                metric.update(preds, targets)
                
                val_pbar.set_postfix({'status': 'processing'})
            except Exception as e:
                print(f"\n⚠️ Validation error: {e}")
                continue
        
        val_pbar.close()

    results = metric.compute()
    current_map = results['map'].item() if 'map' in results else 0.0
    map_50 = results['map_50'].item() if 'map_50' in results else 0.0
    map_75 = results.get('map_75', torch.tensor(0.0)).item()

    # Update scheduler
    scheduler.step()
    early_stop = early_stopper.step(current_map)
    status = early_stopper.get_status()

    # -----------------
    # PRINT RESULTS
    # -----------------
    print(f"\n📊 STAGE 4 - RESULTS EPOCH {epoch+1}:")
    print(f"  Training Loss: {train_loss_avg:.4f}")
    if train_losses_avg:
        for k, v in train_losses_avg.items():
            print(f"    - {k}: {v:.4f}")
    print(f"\n  Validation mAP:")
    print(f"    - mAP@0.5:0.95: {current_map:.4f}")
    print(f"    - mAP@0.5: {map_50:.4f}")
    print(f"    - mAP@0.75: {map_75:.4f}")
    print(f"\n  📊 Early Stopping:")
    print(f"    - Best mAP: {status['best_score']:.4f}")
    print(f"    - No improvement: {status['counter']}/{status['patience']}")
    print(f"    - Remaining: {status['remaining']}")

    # Save best model to Google Drive
    if current_map > best_map:
        best_map = current_map
        best_model_path = os.path.join(SAVE_DIR, "faster_rcnn_best_stage4.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✅ New best model saved to Google Drive! (mAP: {best_map:.4f})")
        print(f"     📁 {best_model_path}")

    if early_stop:
        print("\n" + "="*70)
        print("🛑 EARLY STOPPING TRIGGERED!")
        print(f"  No improvement for {EARLY_STOPPING_PATIENCE} epochs")
        print(f"  Best mAP: {status['best_score']:.4f}")
        print("="*70)
        break

# =========================
# SAVE FINAL MODEL
# =========================
print("\n" + "="*70)
print("SAVING FINAL MODEL - STAGE 4")
print("="*70)

final_model_path = os.path.join(SAVE_DIR, "faster_rcnn_final_stage4.pth")
torch.save(model.state_dict(), final_model_path)

file_size = os.path.getsize(final_model_path) / (1024 * 1024)
print(f"  💾 Final model saved to Google Drive: {final_model_path}")
print(f"  📁 File size: {file_size:.2f} MB")

# Save with mAP in filename
if best_map > 0:
    best_model_path = os.path.join(SAVE_DIR, f"faster_rcnn_best_stage4_mAP_{best_map:.4f}.pth")
    torch.save(model.state_dict(), best_model_path)
    print(f"  💾 Best model saved with mAP in filename: {best_model_path}")

# =========================
# SUMMARY
# =========================
print("\n" + "="*70)
print("✅ STAGE 4 TRAINING COMPLETE!")
print("="*70)

print(f"\n📈 STAGE 4 SUMMARY:")
print(f"  Total epochs: {epoch+1}")
print(f"  Best mAP: {best_map:.4f}")
print(f"  Final mAP: {current_map:.4f}")
print(f"  Improvement from Stage 3: {best_map - 0.2961:.4f}")

print(f"\n📁 Saved files in Google Drive ({SAVE_DIR}):")
print(f"  - faster_rcnn_final_stage4.pth ({file_size:.2f} MB)")
if best_map > 0:
    best_path = os.path.join(SAVE_DIR, "faster_rcnn_best_stage4.pth")
    if os.path.exists(best_path):
        best_size = os.path.getsize(best_path) / (1024 * 1024)
        print(f"  - faster_rcnn_best_stage4.pth ({best_size:.2f} MB) - BEST MODEL 🏆")
    print(f"  - faster_rcnn_best_stage4_mAP_{best_map:.4f}.pth")

print("\n🎯 FINAL CONFIGURATION:")
print("="*70)
print("✅ 1. UNFROZE ONLY layer3 and layer4 (layer1+2 FROZEN)")
print("✅ 2. Learning Rates:")
print("     - Backbone: 0.00002")
print("     - RPN: 0.00005")
print("     - ROI: 0.00005")
print("✅ 3. Optimizer: AdamW")
print("✅ 4. Scheduler: CosineAnnealing")
print("✅ 5. Early Stopping Patience: 6")
print("✅ 6. Total Epochs: 20")
print("✅ 7. Models saved to Google Drive")
print("="*70)

print("\n🏆 FINAL SUMMARY - ALL STAGES:")
print("="*70)
print(f"  Stage 1 (20 epochs): Best mAP = 0.2464")
print(f"  Stage 2 (12 epochs): Best mAP = 0.2927  (+0.0463)")
print(f"  Stage 3 (5 epochs):  Best mAP = 0.2961  (+0.0034)")
print(f"  Stage 4 (20 epochs): Best mAP = {best_map:.4f}")
print("="*70)

print("\n✅ All done! Training completed successfully!")
print("🎯 Best model saved in your Google Drive!")
print("📁 Location: /content/drive/MyDrive/ppe_models/")